# Notebook 12 — Generate DeepSeek Explanations

## Goal
Use the DeepSeek API to generate plain-language explanations for each forecast,
grounded in the retrieved news evidence from Notebook 11.

If `DEEPSEEK_API_KEY` is not set, a deterministic template explanation is used instead.

---

## Prompt rules
- Use **only** the retrieved evidence. Do not invent facts.
- Cite URLs from the evidence.
- If evidence is weak, say so explicitly.
- Explain whether retrieved labor-market news *supports* the forecast direction.
- Temperature = 0.0 (deterministic).

---

## What can go wrong
- DeepSeek API rate limits: add sleep between calls
- API may refuse explanations for certain content: log and use fallback
- Model may hallucinate facts not in the evidence: evaluate in NB 13
- Long evidence blocks may exceed context limits: truncate to top 5 articles

---

## What you must inspect
- Are explanations grounded in the retrieved articles?
- Do any explanations invent numbers or companies not in the evidence?
- Does the explanation correctly state the forecast direction?

In [ ]:
import os
import pathlib, json, datetime, time
import pandas as pd
import numpy as np
import yaml
from tqdm import tqdm

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

EVIDENCE_PATH = DRIVE_ROOT / 'outputs' / 'explanations' / 'retrieved_evidence.parquet'
PREDICTIONS_PATH = DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_news.parquet'
EXPLANATIONS_DIR = DRIVE_ROOT / 'outputs' / 'explanations'

ap_path = DRIVE_ROOT / 'approvals' / '11_rag_retrieval_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 11 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 11 not approved.'

with open(REPO_DIR / 'configs' / 'evidence_config.yaml') as f:
    ev_cfg = yaml.safe_load(f)

# API key check
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
USE_DEEPSEEK = DEEPSEEK_API_KEY is not None

if USE_DEEPSEEK:
    print(f'DeepSeek API key found ({DEEPSEEK_API_KEY[:6]}...). Using DeepSeek API.')
else:
    print('WARNING: DEEPSEEK_API_KEY not set. Using deterministic template fallback.')
    print('Set os.environ["DEEPSEEK_API_KEY"] to use DeepSeek explanations.')

evidence_df = pd.read_parquet(EVIDENCE_PATH)
evidence_df['forecast_date'] = pd.to_datetime(evidence_df['forecast_date'])
evidence_df['article_date'] = pd.to_datetime(evidence_df['article_date'])

preds = pd.read_parquet(PREDICTIONS_PATH)
preds['forecast_date'] = pd.to_datetime(preds['forecast_date'])

print(f'Evidence rows: {len(evidence_df)}')
print(f'Forecasts to explain: {len(preds)}')

## DeepSeek client setup

In [ ]:
if USE_DEEPSEEK:
    from openai import OpenAI
    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=ev_cfg['explanation']['base_url'],
    )
    MODEL_NAME = ev_cfg['explanation']['model']
    TEMPERATURE = ev_cfg['explanation']['temperature']
    MAX_TOKENS = ev_cfg['explanation']['max_tokens']
    print(f'DeepSeek client ready. Model: {MODEL_NAME}, temperature: {TEMPERATURE}')
else:
    client = None
    print('Using template fallback — no API client initialized.')

## Prompt construction and explanation functions

In [ ]:
PROMPT_TEMPLATE = ev_cfg['explanation']['prompt_template']

def build_evidence_block(ev_rows, max_articles=5):
    """Format retrieved articles into a text block for the prompt."""
    rows = ev_rows.head(max_articles)
    lines = []
    for i, row in enumerate(rows.itertuples(), 1):
        lines.append(
            f'{i}. [{row.query_group}] {row.article_date.date()} — {row.title}\n'
            f'   URL: {row.url}'
        )
    return '\n'.join(lines) if lines else 'No articles retrieved for this forecast period.'


def call_deepseek(prompt, client, model, temperature, max_tokens, max_retries=3):
    """Call DeepSeek API with retry logic."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
            else:
                return f'[API error after {max_retries} attempts: {str(e)[:100]}]'


def generate_template_explanation(forecast_month, prediction, n_articles, ev_rows):
    """Deterministic template fallback when no API key is set."""
    direction = 'gain' if prediction > 0 else 'decline'
    top_groups = ev_rows['query_group'].value_counts().head(3).index.tolist() if len(ev_rows) > 0 else []
    top_signals = ', '.join(top_groups) if top_groups else 'none'
    tmpl = ev_cfg['explanation']['fallback']['template']
    return tmpl.format(
        forecast_month=forecast_month,
        prediction=prediction,
        n_articles=n_articles,
        top_signals=top_signals,
    )


def generate_explanation(forecast_date, prediction, ev_rows, client, use_api):
    """Generate an explanation for one forecast."""
    fm = pd.Timestamp(forecast_date).strftime('%Y-%m')
    direction = f'{prediction:+.0f}K ({'positive gain' if prediction > 0 else 'decline'})"
    evidence_block = build_evidence_block(ev_rows)

    if use_api and client is not None:
        prompt = PROMPT_TEMPLATE.format(
            forecast_month=fm,
            prediction=prediction,
            direction=direction,
            forecast_date=pd.Timestamp(forecast_date).strftime('%Y-%m-%d'),
            evidence_block=evidence_block,
        )
        explanation = call_deepseek(
            prompt, client, MODEL_NAME, TEMPERATURE, MAX_TOKENS,
            max_retries=ev_cfg['explanation']['retry_attempts']
        )
        method = 'deepseek'
    else:
        explanation = generate_template_explanation(fm, prediction, len(ev_rows), ev_rows)
        method = 'template'

    cited_urls = ev_rows['url'].tolist() if len(ev_rows) > 0 else []
    return explanation, method, cited_urls

print('Explanation functions defined.')

## Generate explanations for all forecast months

In [ ]:
explanation_rows = []

for _, pred_row in tqdm(preds.iterrows(), total=len(preds), desc='Generating explanations'):
    fd = pred_row['forecast_date']
    prediction = float(pred_row['prediction'])
    actual = float(pred_row['actual'])

    # Get evidence for this forecast date
    ev_rows = evidence_df[evidence_df['forecast_date'] == fd].copy()

    explanation, method, cited_urls = generate_explanation(
        fd, prediction, ev_rows, client, USE_DEEPSEEK
    )

    explanation_rows.append({
        'forecast_date': fd,
        'forecast_month': fd.strftime('%Y-%m'),
        'prediction': prediction,
        'actual': actual,
        'n_articles_retrieved': len(ev_rows),
        'explanation': explanation,
        'explanation_method': method,
        'cited_urls': json.dumps(cited_urls),
    })

    if USE_DEEPSEEK:
        time.sleep(1)  # rate limit

explanations_df = pd.DataFrame(explanation_rows)
print(f'Generated {len(explanations_df)} explanations')
print(f'Method distribution: {explanations_df["explanation_method"].value_counts().to_dict()}')

## Inspect sample explanations

In [ ]:
def show_explanation(df, forecast_month):
    row = df[df['forecast_month'] == forecast_month]
    if len(row) == 0:
        print(f'Forecast month {forecast_month} not found.')
        return
    row = row.iloc[0]
    print(f'\n{"="*70}')
    print(f'Forecast month: {row["forecast_month"]}')
    print(f'Prediction:     {row["prediction"]:+.0f}K  |  Actual: {row["actual"]:+.0f}K')
    print(f'Method:         {row["explanation_method"]}')
    print(f'Articles used:  {row["n_articles_retrieved"]}')
    print(f'\nExplanation:')
    print(row['explanation'])
    print(f'\nCited URLs:')
    for url in json.loads(row['cited_urls']):
        print(f'  {url}')

# Show last 3 forecasts
for fm in explanations_df['forecast_month'].tail(3).values:
    show_explanation(explanations_df, fm)

## Manual Approval Gate

**Before approving:**
1. At least 3 sample explanations look grounded in the retrieved evidence
2. No explanation invents specific numbers or companies not in the evidence
3. Forecast direction (positive/negative) is correctly stated
4. Template explanations (if used) are factually correct

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
out_path = EXPLANATIONS_DIR / 'forecast_explanations.parquet'
explanations_df.to_parquet(out_path, index=False)
print(f'Explanations saved: {out_path}')

approval = {
    'approved': True,
    'notebook': '12_generate_deepseek_explanations.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(EVIDENCE_PATH), str(PREDICTIONS_PATH)],
    'output_artifacts': [str(out_path)],
    'row_counts': {'explanations': int(len(explanations_df))},
    'warnings': [] if USE_DEEPSEEK else ['deepseek_api_not_used_template_fallback'],
    'notes': f'method={explanations_df["explanation_method"].value_counts().to_dict()}'
}
ap_path = DRIVE_ROOT / 'approvals' / '12_deepseek_explanations_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 12 complete. Proceed to 13_evaluate_rag_explanations.ipynb')